Why PPO exists

Core Mechanism

We already know policy gradients and actor-critic, so PPO starts from the problem they have.

1. The problem: policy updates can be too large

Suppose the old policy gives:

π_old(action | state) = 0.50

We collect a trajectory and determine that this action had positive advantage.

A normal policy-gradient update increases its probability:

0.50 → 0.60 → 0.70 → ...

The problem is that a single batch of experience can cause the policy to move too far.

A large policy change can make the new policy very different from the policy that generated the data.

PPO constrains this change.

2. PPO compares old and new policies

For every action from the collected trajectory, calculate:

$$ r_t(\theta) = \frac{\pi_\theta(a_t|s_t)} {\pi_{\text{old}}(a_t|s_t)} $$

This is simply the probability ratio.

Example:

old probability = 0.50
new probability = 0.60

ratio = 0.60 / 0.50
      = 1.2

Interpretation:

ratio = 1 → probability unchanged
ratio > 1 → new policy increased probability
ratio < 1 → new policy decreased probability
3. Combine ratio with advantage

We already calculate an advantage \(A_t\).

If:

A > 0

the action was better than expected, so we want its probability to increase.

If:

A < 0

the action was worse than expected, so we want its probability to decrease.

The basic policy objective is:

$$ r_t A_t $$

Example:

ratio = 1.2
advantage = +2

objective = 1.2 × 2
          = 2.4

So far, this is just importance-weighted policy gradient.

4. PPO clipping

PPO introduces a clipping range:

$$ 1-\epsilon \leq r_t \leq 1+\epsilon $$

Usually:

$$ \epsilon = 0.2 $$

So the useful range is:

0.8 ───────── 1.0 ───────── 1.2

Suppose:

old probability = 0.50
new probability = 0.80

ratio = 1.6

If the advantage is positive, ordinary policy gradient would strongly reward this increase.

PPO clips the ratio:

1.6 → 1.2

The policy therefore doesn't receive additional objective benefit from moving further in that direction.

5. Why the min exists

The actual PPO surrogate objective is:

$$ L^{CLIP} = \mathbb{E} \left[ \min \left( r_tA_t, \operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t \right) \right] $$

The min makes PPO conservative.

Positive advantage

We want:

probability ↑

but not excessively.

Negative advantage

We want:

probability ↓

but not excessively.

So clipping limits how much improvement the optimizer can claim from moving the policy too far.

6. Why PPO keeps the old policy

This is important.

We collect data using:

π_old

Then freeze those probabilities.

For example, suppose our rollout contained:

state = S
action = RIGHT
π_old(RIGHT|S) = 0.40

During optimization, the current policy may change:

π_new(RIGHT|S) = 0.44
π_new(RIGHT|S) = 0.50
π_new(RIGHT|S) = 0.55

We always compare against the original:

0.40

Therefore:

0.44 / 0.40 = 1.10
0.50 / 0.40 = 1.25
0.55 / 0.40 = 1.375

The ratio tells PPO how far the current policy has moved from the policy that generated this data.

That's why we store the old log-probabilities during rollout.

7. PPO training loop

The complete mechanism is:

                OLD POLICY
                    │
                    ▼
             collect rollout
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
       rewards            old log_probs
          │
          ▼
       GAE / advantages
          │
          ▼
    ┌─────────────────┐
    │ PPO optimization│
    └─────────────────┘
          │
          ▼
 current log_probs
          │
          ▼
 ratio = exp(new_log_prob - old_log_prob)
          │
          ▼
      clipping
          │
          ▼
      policy loss
          │
          ▼
      update actor

The critic is trained alongside it using the value targets.

8. One subtle but important point

PPO does not prevent the parameters from changing by more than 20%.

The 0.2 clipping applies to the probability ratio in the PPO objective, not directly to neural-network weights.

That distinction matters.

Yes — the critic \(V(s)\) is independent of the specific action.

It evaluates the state itself:

“How good is it to be in this state?”

So:

V(S1) = 7

doesn't care whether you choose ↑, →, ↓, or ←.

The actor evaluates/chooses actions:

π(↑|S1)
π(→|S1)
π(↓|S1)
π(←|S1)

Then we use the critic's state value as the reference to judge the chosen action.

Critic: state → expected future reward
Actor: state → action probabilities

Example

We are at state S1.

The critic currently predicts:

V(S1) = 5

This means:

"From S1, I currently expect about 5 total future reward."

Now the actor chooses an action.

S1
 ↓
actor chooses RIGHT
 ↓
environment

The environment gives:

reward = +2

and moves us to:

S2
Step 1 — What does the critic think about S2?

The critic looks at S2 and predicts:

V(S2) = 6

So now we have:

S1 --RIGHT / +2--> S2

V(S1) = 5
V(S2) = 6
Step 2 — Calculate what S1 was actually worth

We don't know the true value of S1 yet.

So we create a target:

target = immediate reward + discounted future value

With:

reward = 2
γ = 0.9
V(S2) = 6

we get:

target = 2 + 0.9 × 6
       = 7.4

So we're saying:

"Based on what just happened, S1 looks like it was worth about 7.4, not 5."

Step 3 — Update the critic

Before:

Critic:
V(S1) = 5

Target:

7.4

So the critic's training pushes:

5 → closer to 7.4

That's all the critic update is doing.

Step 4 — What about the actor?

Now we can use this information to judge the action.

The critic expected:

V(S1) = 5

But the observed outcome suggests:

7.4

So:

7.4 - 5 = +2.4

Positive → the outcome was better than the critic expected.

Therefore the actor gets feedback:

"RIGHT turned out better than expected; increase its probability."

The complete flow
S1
 │
 │ critic: "I expect 5"
 ↓
Actor chooses RIGHT
 │
 ↓
Environment
 │
 ├── reward = +2
 ↓
S2
 │
 │ critic: "S2 is worth 6"
 ↓
target = 2 + 0.9×6 = 7.4
 │
 ├──→ Critic: 5 → 7.4
 │
 └──→ Actor: outcome was better than expected
             → increase RIGHT probability

One key distinction: the critic is learning "how valuable is S1?", while the actor is learning "which action should I take from S1?".

Q:
but at starting if critiques state values itself were random how is it trained then purely based on next state which is again a random val? in short

at initialization, the critic can be wrong everywhere. That's okay because it doesn't need a correct critic to start learning.

The key is that the environment reward is the anchor.

Example:

Critic starts:
V(S1) = random 3
V(S2) = random 8

Agent does:

S1 → action → S2

Environment says:

reward = +2

Our first target is:

target = 2 + γ × V(S2)
       = 2 + 0.9 × 8
       = 9.2

Yes, 9.2 is partly wrong because V(S2)=8 was random.

But we still update:

V(S1): 3 → 9.2

Then, crucially, the agent keeps experiencing the environment.

When it eventually reaches an actual terminal reward, there is no future estimate needed:

S9 → terminal
reward = +10

target = 10

That real reward propagates backward through TD updates:

S9 → actual +10
 ↑
S8 → learns from S9
 ↑
S7 → learns from S8
 ↑
...
S1

So the critic starts noisy, but real environmental rewards progressively anchor its estimates.

Let's use a very simple deterministic episode.

Episode
S1 → S2 → S3 → S4 → GOAL
                         +10

Only the goal gives reward:

S1→S2 = 0
S2→S3 = 0
S3→S4 = 0
S4→GOAL = +10

At the beginning:

V(S1) = 0
V(S2) = 0
V(S3) = 0
V(S4) = 0
Episode 1

At the final transition:

S4 → GOAL
reward = +10

There is no future state value because we're terminal.

So:

target V(S4) = 10

S4 learns:

0 → 10

Already, the reward has made a huge change.

Next time we encounter S3

Now:

S3 → S4
reward = 0

But S4 is no longer zero.

V(S4) = 10

Therefore:

target(S3)
= 0 + γ × 10

If γ = 0.9:

target(S3) = 9

So:

V(S3): 0 → 9
Then S2

Now S3 has information:

V(S3) = 9

So:

target(S2)
= 0 + 0.9 × 9
= 8.1

Therefore:

V(S2): 0 → 8.1

Then:

V(S1)
≈ 0.9 × 8.1
= 7.29

So look what happened:

              +10
               ↓
S1 → S2 → S3 → S4 → GOAL
↑     ↑     ↑     ↑
7.29  8.1   9    10
The reward didn't need to be given at every step.

The value estimate itself carries the information backward.

That's why bootstrapping is powerful.

But there's an important correction

It isn't necessarily:

one episode
→ perfectly propagate +10 all the way back

With neural networks, stochastic environments, imperfect exploration, and TD updates, it happens gradually across many experiences.

And that's where your intuition was right:

If the reward is extremely sparse AND the agent rarely reaches it, learning can be painfully slow.

That's a real RL problem called the sparse-reward / credit-assignment problem.

But once the agent encounters the reward, the critic can propagate its information through the value estimates.

So the chain is:

REAL REWARD
    ↓
V(S4)
    ↓
V(S3)
    ↓
V(S2)
    ↓
V(S1)

Not because the critic magically knows the answer, but because each corrected state becomes a better teaching signal for the previous state.

That's the piece that makes TD learning work despite sparse rewards.